In [1]:
!pip install -qq bitsandbytes peft trl math_verify

# Game Theory GRPO


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_type=torch.bfloat16
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Dataset for Game Theory

In [3]:
ds = load_dataset("Alogotron/GameTheory-Formulator")['train']
trainTest = ds.train_test_split(test_size=0.1)
train, test = trainTest['train'], trainTest['test']

In [4]:
train

Dataset({
    features: ['id', 'domain', 'scenario', 'game_type', 'game_subtype', 'players', 'strategies', 'payoffs', 'information', 'solution_concept', 'formulation_steps', 'solution', 'interpretation', 'difficulty', 'tags'],
    num_rows: 1093
})

In [5]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a game theory scenario, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)
def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["scenario"]},
        ],
    }

train_dataset = train.map(make_conversation)
test_dataset = test.map(make_conversation)

Map:   0%|          | 0/1093 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

In [6]:
# Re-initialize dataset splits from original source
train, test = trainTest['train'], trainTest['test']

# Map with system prompt to create 'prompt' column
train_dataset = train.map(make_conversation)
test_dataset = test.map(make_conversation)

# Safely filter columns to keep required columns for GRPO and reward functions
keep_columns = ['prompt', 'players', "payoffs", "strategies"]
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_columns])
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in keep_columns])

print("Rebuilt datasets successfully!")
print("Train columns:", train_dataset.column_names)


Rebuilt datasets successfully!
Train columns: ['players', 'strategies', 'payoffs', 'prompt']


In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.01,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

In [8]:
!pip install -qq math-verify

In [9]:
import re


def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content) for content in completion_contents]
    rewards_list = [1.0 if match else 0.0 for match in matches]
    return [1.0 if match else 0.0 for match in matches]

In [ ]:
from math_verify import parse, LatexExtractionConfig, verify


def accuracy_reward_players(completions, **kwargs):
    """Reward function that verifies if players, payoffs, and strategies are correctly identified in the answer."""
    players_list = kwargs['players']
    payoffs_list = kwargs['payoffs']
    strategies_list = kwargs['strategies']

    completion_contents = [completion[0]["content"] for completion in completions]
    rewards = []

    for content, gold_players, gold_payoffs, gold_strategies in zip(completion_contents, players_list, payoffs_list, strategies_list):
        # Extract the text inside the <answer>...</answer> tags if present, otherwise search the whole content
        answer_match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        answer_text = answer_match.group(1) if answer_match else content

        score = 0.0
        components_checked = 0

        # 1. Verify Players
        if gold_players:
            components_checked += 1
            gold_players_parsed = parse(gold_players, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
            answer_parsed = parse(answer_text, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
            try:
                if verify(answer_parsed, gold_players_parsed) or str(gold_players).lower() in answer_text.lower():
                    score += 1.0
            except Exception:
                pass

        # 2. Verify Payoffs
        if gold_payoffs:
            components_checked += 1
            gold_payoffs_parsed = parse(gold_payoffs, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
            answer_parsed = parse(answer_text, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
            try:
                if verify(answer_parsed, gold_payoffs_parsed) or str(gold_payoffs).lower() in answer_text.lower():
                    score += 1.0
            except Exception:
                pass

        # 3. Verify Strategies
        if gold_strategies:
            components_checked += 1
            if str(gold_strategies).lower() in answer_text.lower():
                score += 1.0

        # Calculate final average accuracy score for this sample
        if components_checked > 0:
            rewards.append(score / components_checked)
        else:
            rewards.append(1.0)

    return rewards

In [11]:
from trl import GRPOTrainer, GRPOConfig

# Configure with settings to prevent mode collapse
training_args_stable = GRPOConfig(
    output_dir="Qwen/Qwen2.5-1.5B-Instruct-game-theory",
    learning_rate=1e-5,
    temperature=0.6,
    remove_unused_columns=False,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    bf16=True,
    max_completion_length=128,
    num_generations=4,
    report_to="none",
    logging_steps=10,
    push_to_hub=False,
    save_strategy="steps",
    save_steps=10,
)

grpo_trainer_stable = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, accuracy_reward_players],
    args=training_args_stable,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

Stable GRPOTrainer initialized. You can now call `grpo_trainer_stable.train()`!


In [12]:
grpo_trainer_stable.train()

Step,Training Loss
10,0.074956
20,0.068771
30,0.088649
40,0.081431
50,0.094021
60,0.095680


TrainOutput(global_step=68, training_loss=0.08571113646030426, metrics={'train_runtime': 697.4127, 'train_samples_per_second': 3.134, 'train_steps_per_second': 0.098, 'total_flos': 0.0, 'train_loss': 0.08571113646030426, 'num_tokens': 2870073.0, 'completions/mean_length': 94.8603515625, 'completions/min_length': 22.5, 'completions/max_length': 128.0, 'completions/clipped_ratio': 0.3837890625, 'completions/mean_terminated_length': 74.127685546875, 'completions/min_terminated_length': 22.5, 'completions/max_terminated_length': 126.875, 'rewards/format_reward/mean': 0.4951171875, 'rewards/format_reward/std': 0.5000358372926712, 'rewards/accuracy_reward_players/mean': 0.0, 'rewards/accuracy_reward_players/std': 0.0, 'reward': 0.4951171875, 'reward_std': 0.5000358372926712, 'frac_reward_zero_std': 0.1953125, 'entropy': 0.5218212341424078, 'clip_ratio/low_mean': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/region_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_max': 0.0, 'step_time'

In [15]:
del model, tokenizer
torch.cuda.empty_cache()
import gc
gc.collect()

7455

In [16]:
model_name = "nickkats1/Qwen2.5-1.5B-Instruct-game-theory"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 2.19MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [17]:
from datasets import load_dataset

test_data = load_dataset("Alogotron/GameTheory-Bench")['train']
test_data

Dataset({
    features: ['id', 'category', 'subcategory', 'difficulty', 'problem', 'solution', 'answer', 'verification', 'game_type', 'information', 'players', 'tags'],
    num_rows: 2913
})

In [52]:
sample1 = test_data['problem'][0]

In [ ]:
import time


def generate_with_reasoning(prompt):
    if isinstance(prompt, list):
        prompt_text = " ".join(entry["content"] for entry in prompt if isinstance(entry, dict) and "content" in entry)
    else:
        prompt_text = str(prompt)


    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)


    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=500)
    end_time = time.time()


    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)


    inference_duration = end_time - start_time


    num_input_tokens = inputs["input_ids"].shape[1]
    num_generated_tokens = output_ids.shape[1] - num_input_tokens

    return generated_text, inference_duration, num_generated_tokens

In [21]:
sample1 = test_data['problem'][0]

In [22]:
generated_text, inference_duration, num_generated_tokens = generate_with_reasoning(sample1)
print(generated_text)

Consider the following Prisoner's Dilemma game:

Player 1 \ Player 2 | Cooperate | Defect
--- | --- | ---
Cooperate | (-1,-1) | (-3,0)
Defect | (0,-3) | (-2,-2)

Find all Nash Equilibria and explain the strategic structure. To find the Nash equilibria in this Prisoner's Dilemma game, we need to identify the strategies where neither player can benefit by unilaterally changing their strategy.

Let's analyze the possible outcomes for each combination of strategies:

1. **(Cooperate, Cooperate)**:
   - Player 1 gets (-1, -1).
   - Player 2 gets (-1, -1).

2. **(Cooperate, Defect)**:
   - Player 1 gets (-3, 0).
   - Player 2 gets (0, -3).

3. **(Defect, Cooperate)**:
   - Player 1 gets (0, -3).
   - Player 2 gets (-2, -2).

4. **(Defect, Defect)**:
   - Player 1 gets (-2, -2).
   - Player 2 gets (-2, -2).

A Nash equilibrium occurs when both players are playing a strategy that is not dominated by any other strategy available to them. Let's examine each case:

- In **(Cooperate, Cooperate)**

In [36]:
solution1 = test_data[0]
solution1

{'id': 'gt-2x2-classic-0001',
 'category': 'normal_form_2x2',
 'subcategory': "classic_prisoner's_dilemma",
 'difficulty': 'easy',
 'problem': "Consider the following Prisoner's Dilemma game:\n\nPlayer 1 \\ Player 2 | Cooperate | Defect\n--- | --- | ---\nCooperate | (-1,-1) | (-3,0)\nDefect | (0,-3) | (-2,-2)\n\nFind all Nash Equilibria and explain the strategic structure.",
 'solution': "This game has 1 pure strategy Nash Equilibrium.\n\nTo verify (Down, Right) is a Nash Equilibrium, we check each player's incentive to deviate:\n- Player 1 currently plays Down, getting payoff -2. If Player 1 switches to Up (while Player 2 stays at Right), payoff drops to -3. Player 1 has no incentive to deviate. ✓\n- Player 2 currently plays Right, getting payoff -2. If Player 2 switches to Left (while Player 1 stays at Down), payoff drops to -3. Player 2 has no incentive to deviate. ✓\nTherefore (Down, Right) is a Nash Equilibrium with payoffs (-2,-2).\n\nThere are no additional mixed strategy equili